### **Notebook 2: Feature Engineering & Time Series Preparation**

**Goal:**

- Create time-based features (day, week, lag values)

- Prepare data for machine learning forecasting

- Split data into train and test sets

**Objective:**

Transform raw trip data into a model-ready time series dataset.

In [ ]:
# import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

In [ ]:
# load cleaned file saved in notebook 1
df = pd.read_csv('/content/uber_raw_with_tripdate.csv', parse_dates=['trip_date'])
df = df.sort_values(['dispatching_base_number','trip_date']).reset_index(drop=True)
df.head()

,dispatching_base_number,date,active_vehicles,trips,trip_date
0,B02512,1/1/2015,190,1132,2015-01-01
1,B02512,1/2/2015,175,875,2015-01-02
2,B02512,1/3/2015,173,1088,2015-01-03
3,B02512,1/4/2015,147,791,2015-01-04
4,B02512,1/5/2015,194,984,2015-01-05


In [ ]:
# check for missing values
df.isnull().sum()

,0
dispatching_base_number,0
date,0
active_vehicles,0
trips,0
trip_date,0


In [ ]:
# add calendar features
df['day_of_week'] = df['trip_date'].dt.dayofweek       # 0=Mon … 6=Sun
df['day_name'] = df['trip_date'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)
df['month'] = df['trip_date'].dt.month
df.head()

,dispatching_base_number,date,active_vehicles,trips,trip_date,day_of_week,day_name,is_weekend,month
0,B02512,1/1/2015,190,1132,2015-01-01,3,Thursday,0,1
1,B02512,1/2/2015,175,875,2015-01-02,4,Friday,0,1
2,B02512,1/3/2015,173,1088,2015-01-03,5,Saturday,1,1
3,B02512,1/4/2015,147,791,2015-01-04,6,Sunday,1,1
4,B02512,1/5/2015,194,984,2015-01-05,0,Monday,0,1


In [ ]:
# label encode base column
le = LabelEncoder()
df['base_encoded'] = le.fit_transform(df['dispatching_base_number'])
df[['dispatching_base_number','base_encoded']].head()

,dispatching_base_number,base_encoded
0,B02512,0
1,B02512,0
2,B02512,0
3,B02512,0
4,B02512,0


**Insight:** Encoding creates a numeric category for each base → required for the unified ML model.

In [ ]:
# create lag, rolling and ratio features per base
bases = df['dispatching_base_number'].unique()
feature_frames = []

for base in bases:
    # filter for one base and sort
    temp = df[df['dispatching_base_number'] == base].copy()
    temp = temp.sort_values('trip_date').reset_index(drop=True)

    # lag features
    temp['trips_lag_1'] = temp['trips'].shift(1)
    temp['trips_lag_7'] = temp['trips'].shift(7)
    temp['trips_lag_14'] = temp['trips'].shift(14)

    # rolling means
    temp['roll_mean_7'] = temp['trips'].shift(1).rolling(7).mean()
    temp['roll_std_7']  = temp['trips'].shift(1).rolling(7).std()

    temp['roll_mean_14'] = temp['trips'].shift(1).rolling(14).mean()
    temp['roll_std_14']  = temp['trips'].shift(1).rolling(14).std()

    # ratio feature
    temp['trips_per_vehicle'] = temp['trips'] / temp['active_vehicles'].replace(0, np.nan)
    temp['trips_per_vehicle'] = temp['trips_per_vehicle'].fillna(0)

    # append
    feature_frames.append(temp)

# combine all bases
feat_df = pd.concat(feature_frames).reset_index(drop=True)
feat_df.head(20)

,dispatching_base_number,date,active_vehicles,trips,trip_date,day_of_week,day_name,is_weekend,month,base_encoded,trips_lag_1,trips_lag_7,trips_lag_14,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,trips_per_vehicle
0,B02512,1/1/2015,190,1132,2015-01-01,3,Thursday,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.957895
1,B02512,1/2/2015,175,875,2015-01-02,4,Friday,0,1,0,1132.0,NaN,NaN,NaN,NaN,NaN,NaN,5.000000
2,B02512,1/3/2015,173,1088,2015-01-03,5,Saturday,1,1,0,875.0,NaN,NaN,NaN,NaN,NaN,NaN,6.289017
3,B02512,1/4/2015,147,791,2015-01-04,6,Sunday,1,1,0,1088.0,NaN,NaN,NaN,NaN,NaN,NaN,5.380952
4,B02512,1/5/2015,194,984,2015-01-05,0,Monday,0,1,0,791.0,NaN,NaN,NaN,NaN,NaN,NaN,5.072165
5,B02512,1/6/2015,218,1314,2015-01-06,1,Tuesday,0,1,0,984.0,NaN,NaN,NaN,NaN,NaN,NaN,6.027523
6,B02512,1/7/2015,217,1446,2015-01-07,2,Wednesday,0,1,0,1314.0,NaN,NaN,NaN,NaN,NaN,NaN,6.663594
7,B02512,1/8/2015,238,1772,2015-01-08,3,Thursday,0,1,0,1446.0,1132.0,NaN,1090.000000,232.931320,NaN,NaN,7.445378
8,B02512,1/9/2015,224,1560,2015-01-09,4,Friday,0,1,0,1772.0,875.0,NaN,1181.428571,348.900109,NaN,NaN,6.964286
9,B02512,1/10/2015,206,1646,2015-01-10,5,Saturday,1,1,0,1560.0,1088.0,NaN,1279.285714,344.667334,NaN,NaN,7.990291


**INSIGHT :**
Lag & rolling features introduce NaN for first few rows → normal.

In [ ]:
# drop rows where lag 1 or rolling 7 are null
feat_df = feat_df.dropna(subset=['trips_lag_1','roll_mean_7']).reset_index(drop=True)
feat_df.isnull().sum()

,0
dispatching_base_number,0
date,0
active_vehicles,0
trips,0
trip_date,0
day_of_week,0
day_name,0
is_weekend,0
month,0
base_encoded,0


In [ ]:
# sort for consistency
feat_df = feat_df.sort_values(['trip_date','dispatching_base_number']).reset_index(drop=True)
feat_df.head()

,dispatching_base_number,date,active_vehicles,trips,trip_date,day_of_week,day_name,is_weekend,month,base_encoded,trips_lag_1,trips_lag_7,trips_lag_14,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,trips_per_vehicle
0,B02512,1/8/2015,238,1772,2015-01-08,3,Thursday,0,1,0,1446.0,1132.0,NaN,1090.000000,232.931320,NaN,NaN,7.445378
1,B02598,1/8/2015,1070,10050,2015-01-08,3,Thursday,0,1,1,8397.0,6903.0,NaN,6476.714286,1281.214231,NaN,NaN,9.392523
2,B02617,1/8/2015,1463,13462,2015-01-08,3,Thursday,0,1,2,11528.0,9537.0,NaN,9221.428571,1680.728300,NaN,NaN,9.201640
3,B02682,1/8/2015,1135,10416,2015-01-08,3,Thursday,0,1,3,9078.0,7679.0,NaN,7056.714286,1361.837694,NaN,NaN,9.177093
4,B02764,1/8/2015,3831,33802,2015-01-08,3,Thursday,0,1,4,29949.0,29421.0,NaN,25105.857143,4637.816959,NaN,NaN,8.823284


In [ ]:
# final modeling dataset preview
feat_df[['trip_date','dispatching_base_number','trips','trips_lag_1','roll_mean_7','base_encoded']].head(10)

,trip_date,dispatching_base_number,trips,trips_lag_1,roll_mean_7,base_encoded
0,2015-01-08,B02512,1772,1446.0,1090.000000,0
1,2015-01-08,B02598,10050,8397.0,6476.714286,1
2,2015-01-08,B02617,13462,11528.0,9221.428571,2
3,2015-01-08,B02682,10416,9078.0,7056.714286,3
4,2015-01-08,B02764,33802,29949.0,25105.857143,4
5,2015-01-08,B02765,1911,1704.0,1356.857143,5
6,2015-01-09,B02512,1560,1772.0,1181.428571,0
7,2015-01-09,B02598,9538,10050.0,6926.285714,1
8,2015-01-09,B02617,13165,13462.0,9782.142857,2
9,2015-01-09,B02682,10477,10416.0,7447.714286,3


In [ ]:
# time-based train-test split
unique_dates = sorted(feat_df['trip_date'].unique())
cutoff = unique_dates[int(len(unique_dates)*0.8)]

train_df = feat_df[feat_df['trip_date'] <= cutoff]
test_df  = feat_df[feat_df['trip_date']  > cutoff]

train_df.shape, test_df.shape, cutoff

((252, 18), (60, 18), Timestamp('2015-02-18 00:00:00'))

In [ ]:
# save processed datasets
feat_df.to_csv('/content/uber_daily_processed.csv', index=False)
train_df.to_csv('/content/uber_train.csv', index=False)
test_df.to_csv('/content/uber_test.csv', index=False)

"/content/uber_daily_processed.csv saved"

'/content/uber_daily_processed.csv saved'

**Summary**

- Created lag and rolling features

- Structured data per dispatching base

- Ensured correct time order

- Final dataset ready for model training

Features finalized for modeling in **Notebook 3**